# Prediksi Turnover Karyawan — Pipeline Training dari CSV sampai Model Siap Pakai

**Tujuan:** memprediksi apakah seorang karyawan berpotensi keluar (`target_turnover = 1`) berdasarkan data `hr_turnover.csv`, lalu menyimpan model yang bisa dipakai ulang.

## Alur kerja

| # | Tahap | Yang dikerjakan |
|---|-------|-----------------|
| 1 | Load & audit data | baca CSV, cek tipe, missing value, duplikat, nilai tidak masuk akal |
| 2 | EDA | distribusi target, hubungan tiap fitur dengan target, uji asosiasi |
| 3 | Keputusan fitur | buang ID, tangani tanggal, cek redundansi & risiko *leakage* |
| 4 | Feature engineering | fitur turunan dari tanggal, usia, gaji |
| 5 | Split data | train / test **stratified**, test disimpan dan baru dibuka di akhir |
| 6 | Preprocessing pipeline | imputasi, scaling, one-hot, semuanya di dalam `Pipeline` |
| 7 | Baseline & bandingkan model | Dummy vs Logistic Regression vs Random Forest vs Gradient Boosting (cross-validation) |
| 8 | Hyperparameter tuning | `RandomizedSearchCV` pada model terbaik |
| 9 | Pemilihan threshold | dipilih dari prediksi *out-of-fold* di data train, bukan dari test |
| 10 | Evaluasi final | satu kali di data test: ROC, PR, calibration, confusion matrix, interval kepercayaan |
| 11 | Interpretasi | permutation importance & partial dependence |
| 12 | Simpan & inferensi | simpan model + metadata, lalu contoh prediksi data baru |

## Prinsip yang dijaga
- **Tidak ada data leakage:** semua langkah yang "belajar" dari data (imputer, scaler, encoder) dilatih hanya di data train lewat `Pipeline`.
- **Test set disentuh sekali**, hanya untuk evaluasi akhir.
- **Metrik sesuai masalah:** ROC-AUC & PR-AUC untuk membandingkan model, bukan accuracy saja.
- **Reproducible:** seed tetap, versi library dan parameter dicatat di metadata model.

> **Kebutuhan library:** `pip install pandas numpy scikit-learn matplotlib seaborn scipy joblib` (scikit-learn ≥ 1.2).

## 0. Import & Konfigurasi

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn

from scipy.stats import chi2_contingency, mannwhitneyu, loguniform, randint
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
    accuracy_score, average_precision_score, classification_report,
    f1_score, make_scorer, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV, RepeatedStratifiedKFold, StratifiedKFold,
    cross_val_predict, cross_validate, train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
np.random.seed(SEED)

CANDIDATE_PATHS = ["hr_turnover.csv", "../data/hr_turnover.csv", "/mnt/user-data/uploads/hr_turnover.csv"]
DATA_PATH = next((Path(p) for p in CANDIDATE_PATHS if Path(p).exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "hr_turnover.csv tidak ditemukan. "
        "atau ubah CANDIDATE_PATHS."
    )

TARGET = "target_turnover"
ID_COL = "employee_id"
DATE_COLS = ["tanggal_masuk", "tanggal_promosi"]
REF_DATE = pd.Timestamp("2025-01-01")   
TEST_SIZE = 0.20
REFIT_ON_FULL_DATA = True           
PROJECT_ROOT = Path.cwd().parent
ARTIFACT_DIR = PROJECT_ROOT / "models/v1" 
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

print("Data      :", DATA_PATH)
print("pandas    :", pd.__version__)
print("sklearn   :", sklearn.__version__)

## 1. Load & Audit Data

In [ ]:
df_raw = pd.read_csv(DATA_PATH, parse_dates=DATE_COLS)
print("Ukuran data:", df_raw.shape)
df_raw.head()

In [ ]:
print("=== Tipe data ===")
print(df_raw.dtypes.to_string())

print("\n=== Missing value ===")
miss = df_raw.isna().sum()
print(miss[miss > 0].to_string() if miss.any() else "tidak ada")

print("\n=== Distribusi target ===")
dist = df_raw[TARGET].value_counts().rename("jumlah").to_frame()
dist["persen"] = (dist["jumlah"] / len(df_raw) * 100).round(1)
print(dist.to_string())

print("\n=== Ringkasan numerik ===")
df_raw.describe().T.round(2)

In [ ]:
usia_masuk = df_raw["usia"] - df_raw["masa_kerja_bulan"] / 12

cek = pd.Series({
    "baris duplikat": int(df_raw.duplicated().sum()),
    "employee_id unik": bool(df_raw[ID_COL].is_unique),
    "usia di luar 17-70": int((~df_raw["usia"].between(17, 70)).sum()),
    "gaji <= 0": int((df_raw["gaji_bulanan"] <= 0).sum()),
    "jam lembur < 0": int((df_raw["jam_lembur_bulanan"] < 0).sum()),
    "usia saat masuk < 17": int((usia_masuk < 17).sum()),
    "tanggal_promosi sebelum tanggal_masuk": int((df_raw["tanggal_promosi"] < df_raw["tanggal_masuk"]).sum()),
    "tanggal masuk setelah REF_DATE": int((df_raw["tanggal_masuk"] > REF_DATE).sum()),
    "tanggal promosi setelah REF_DATE": int((df_raw["tanggal_promosi"] > REF_DATE).sum()),
})
print(cek.to_string())

q1, q3 = df_raw["gaji_bulanan"].quantile([0.25, 0.75])
batas = q3 + 1.5 * (q3 - q1)
print(f"\nGaji di atas batas IQR (> {batas:,.0f}): {(df_raw['gaji_bulanan'] > batas).sum()} baris → distribusi right-skewed, ditangani dengan log")

In [ ]:
bulan_dari_tanggal = (
    (REF_DATE.year - df_raw["tanggal_masuk"].dt.year) * 12
    + (REF_DATE.month - df_raw["tanggal_masuk"].dt.month)
)
selisih = bulan_dari_tanggal - df_raw["masa_kerja_bulan"]
print("Selisih (bulan dihitung dari tanggal - masa_kerja_bulan):")
print(selisih.value_counts().sort_index().to_string())

**Catatan hasil audit**
- Data bersih: tidak ada duplikat, `employee_id` unik, tidak ada nilai mustahil.
- Satu-satunya missing value ada di `tanggal_promosi`. Kosong di sini berarti **belum pernah dipromosikan**, jadi harus diperlakukan sebagai informasi (bukan sekadar diimputasi), lewat fitur `ada_promosi`.
- `masa_kerja_bulan` konsisten dengan `tanggal_masuk` terhadap tanggal acuan ±1 Januari 2025 (selisih hanya 0/1 bulan akibat pembulatan). Artinya kolom tanggal **tidak menambah informasi baru** selain lewat fitur turunan promosi. Karena itu tanggal mentah tidak dipakai langsung sebagai fitur.
- Target cukup seimbang (≈36% positif), jadi **tidak perlu** SMOTE/oversampling maupun `class_weight`. Ketimpangan ringan ditangani lewat metrik yang tepat (PR-AUC) dan pemilihan threshold (tahap 9). Tanpa class weighting, probabilitas model juga tetap terkalibrasi sehingga bisa dibaca sebagai skor risiko.

## 2. Exploratory Data Analysis (EDA)

In [ ]:
df_eda = df_raw.assign(ada_promosi=df_raw["tanggal_promosi"].notna().map({True: "Ya", False: "Tidak"}))
base_rate = df_eda[TARGET].mean()

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.ravel(), ["status_kerja", "pendidikan", "departemen", "ada_promosi"]):
    g = df_eda.groupby(col)[TARGET].agg(["mean", "count"]).sort_values("mean", ascending=False)
    sns.barplot(x=g.index.astype(str), y=g["mean"].values, ax=ax, color="#4C78A8")
    ax.axhline(base_rate, ls="--", color="crimson", lw=1, label=f"rata-rata total {base_rate:.1%}")
    for i, (m, n) in enumerate(zip(g["mean"], g["count"])):
        ax.text(i, m + 0.01, f"{m:.0%}\n(n={n})", ha="center", va="bottom", fontsize=9)
    ax.set_ylim(0, g["mean"].max() + 0.17)
    ax.set_title(f"Rasio turnover per {col}")
    ax.set_xlabel("")
    ax.set_ylabel("rasio turnover")
    ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
num_cols_eda = ["usia", "masa_kerja_bulan", "jam_lembur_bulanan", "gaji_bulanan"]

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, col in zip(axes.ravel(), num_cols_eda):
    sns.histplot(data=df_raw, x=col, hue=TARGET, stat="density", common_norm=False,
                 element="step", bins=30, ax=ax, palette={0: "#4C78A8", 1: "#E45756"})
    ax.set_title(f"Distribusi {col} berdasarkan target")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bins = pd.cut(df_raw["jam_lembur_bulanan"], [-1, 5, 10, 20, 30, 40, 100],
              labels=["0-5", "6-10", "11-20", "21-30", "31-40", ">40"])
g = df_raw.groupby(bins, observed=True)[TARGET].agg(["mean", "count"])
sns.barplot(x=g.index.astype(str), y=g["mean"].values, ax=axes[0], color="#F58518")
for i, (m, n) in enumerate(zip(g["mean"], g["count"])):
    axes[0].text(i, m + 0.01, f"{m:.0%}\n(n={n})", ha="center", va="bottom", fontsize=9)
axes[0].set_ylim(0, 1.0)
axes[0].set_title("Rasio turnover per rentang jam lembur bulanan")
axes[0].set_xlabel("jam lembur / bulan")
axes[0].set_ylabel("rasio turnover")

corr = df_raw[["usia", "masa_kerja_bulan", "jam_lembur_bulanan", "gaji_bulanan", TARGET]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Korelasi (Pearson)")
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for col in ["status_kerja", "pendidikan", "departemen", "ada_promosi"]:
    chi2, p, _, _ = chi2_contingency(pd.crosstab(df_eda[col], df_eda[TARGET]))
    rows.append({"fitur": col, "uji": "Chi-square", "p_value": p})

for col in num_cols_eda:
    a = df_raw.loc[df_raw[TARGET] == 1, col]
    b = df_raw.loc[df_raw[TARGET] == 0, col]
    _, p = mannwhitneyu(a, b)
    rows.append({"fitur": col, "uji": "Mann-Whitney U", "p_value": p})

asosiasi = pd.DataFrame(rows)
asosiasi["signifikan (p<0.05)"] = asosiasi["p_value"] < 0.05
asosiasi["p_value"] = asosiasi["p_value"].round(4)
asosiasi.sort_values("p_value").reset_index(drop=True)

**Membaca hasil EDA:** lihat tabel uji asosiasi di atas untuk fitur mana yang benar-benar berkaitan dengan target. Yang menonjol: **jam lembur** paling kuat dan bersifat tidak linear (turnover melonjak tajam pada lembur tinggi), sedangkan status kerja dan gaji memiliki hubungan lebih lemah. Karena hubungannya tidak linear, model berbasis pohon layak dibandingkan dengan model linear.

## 3–4. Keputusan Fitur & Feature Engineering

| Kolom | Keputusan | Alasan |
|-------|-----------|--------|
| `employee_id` | **dibuang** | pengenal unik, tidak punya makna prediktif, bisa memicu overfitting |
| `tanggal_masuk`, `tanggal_promosi` | **diganti fitur turunan** | tanggal mentah tidak generalisasi ke data baru |
| `masa_kerja_bulan` | dipertahankan | sudah numerik dan informatif |
| `gaji_bulanan` | diganti `log_gaji` | distribusi right-skewed |
| `departemen`, `status_kerja`, `pendidikan` | one-hot | kategorik nominal |

**Fitur turunan** (semua dihitung per baris tanpa "belajar" dari data lain, sehingga aman dilakukan sebelum split):
- `ada_promosi` — 0/1, menangkap makna dari `tanggal_promosi` yang kosong
- `bulan_sejak_promosi` — jarak promosi terakhir ke tanggal acuan (kosong bila belum promosi)
- `bulan_hingga_promosi` — lama dari masuk sampai promosi (kosong bila belum promosi)
- `usia_saat_masuk` — `usia − masa_kerja/12`
- `log_gaji` — `log(1 + gaji)`

In [ ]:
NUM_COLS = [
    "usia", "masa_kerja_bulan", "jam_lembur_bulanan", "log_gaji", "usia_saat_masuk",
    "ada_promosi", "bulan_sejak_promosi", "bulan_hingga_promosi",
]
CAT_COLS = ["departemen", "status_kerja", "pendidikan"]
FEATURES = NUM_COLS + CAT_COLS


def buat_fitur(d: pd.DataFrame, ref_date: pd.Timestamp = REF_DATE) -> pd.DataFrame:
    """Ubah data mentah (kolom seperti CSV) menjadi fitur model.
    Fungsi yang sama dipakai saat training dan saat inferensi data baru."""
    d = d.copy()
    masuk = pd.to_datetime(d["tanggal_masuk"])
    promosi = pd.to_datetime(d["tanggal_promosi"])

    d["ada_promosi"] = promosi.notna().astype(int)
    d["bulan_sejak_promosi"] = (ref_date - promosi).dt.days / 30.44
    d["bulan_hingga_promosi"] = (promosi - masuk).dt.days / 30.44
    d["usia_saat_masuk"] = d["usia"] - d["masa_kerja_bulan"] / 12
    d["log_gaji"] = np.log1p(d["gaji_bulanan"])

    fitur = d[FEATURES].copy()
    fitur[NUM_COLS] = fitur[NUM_COLS].astype(float)
    return fitur


X = buat_fitur(df_raw)
y = df_raw[TARGET].astype(int)

print("Bentuk X:", X.shape, "| jumlah fitur:", len(FEATURES))
print("Missing per fitur (hanya fitur promosi yang wajar kosong):")
print(X.isna().sum()[X.isna().sum() > 0].to_string())
X.head()

## 5. Split Data (Stratified Hold-out)

80% train dan 20% **test**. `stratify=y` menjaga proporsi target sama di kedua bagian. Test set **tidak dipakai** untuk memilih model, tuning, maupun threshold. Semua keputusan diambil dengan cross-validation di data train.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

ringkas = pd.DataFrame({
    "baris": [len(X_train), len(X_test)],
    "rasio turnover": [y_train.mean().round(3), y_test.mean().round(3)],
}, index=["train", "test"])
ringkas

## 6. Preprocessing Pipeline

- **Numerik:** imputasi median → `StandardScaler` (penting untuk Logistic Regression, tidak merugikan model pohon)
- **Kategorik:** `OneHotEncoder(handle_unknown="ignore")` agar kategori baru saat inferensi tidak membuat error

Preprocessor ini dibungkus bersama model dalam satu `Pipeline`, sehingga di setiap fold cross-validation ia hanya belajar dari data training fold tersebut (mencegah leakage).

In [ ]:
def make_preprocessor() -> ColumnTransformer:
    numeric = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical = OneHotEncoder(handle_unknown="ignore")
    return ColumnTransformer([
        ("num", numeric, NUM_COLS),
        ("cat", categorical, CAT_COLS),
    ])


make_preprocessor()

## 7. Baseline & Perbandingan Model (Cross-Validation)

Model dibandingkan dengan **5-fold × 3 pengulangan stratified CV** di data train. Metrik utama:
- **ROC-AUC**: kemampuan membedakan yang keluar vs bertahan, tidak bergantung threshold
- **PR-AUC**: fokus pada kelas positif (karyawan yang keluar)
- **Recall / Precision / F1** pada threshold default 0,5, sebagai gambaran awal

`DummyClassifier` menjadi **baseline**: model apa pun yang tidak jauh melampauinya tidak layak dipakai.

In [ ]:
models = {
    "Baseline (Dummy)": DummyClassifier(strategy="prior"),
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=3, n_jobs=1, random_state=SEED),
    "Gradient Boosting (Hist)": HistGradientBoostingClassifier(random_state=SEED),
}

scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "recall": make_scorer(recall_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
}
cv_repeat = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)

cv_mean, cv_text = {}, {}
for name, est in models.items():
    pipe = Pipeline([("prep", make_preprocessor()), ("clf", est)])
    res = cross_validate(pipe, X_train, y_train, cv=cv_repeat, scoring=scoring, n_jobs=-1)
    cv_mean[name] = {m: res[f"test_{m}"].mean() for m in scoring}
    cv_text[name] = {m: f"{res[f'test_{m}'].mean():.3f} ± {res[f'test_{m}'].std():.3f}" for m in scoring}

cv_mean = pd.DataFrame(cv_mean).T.sort_values("roc_auc", ascending=False)
cv_table = pd.DataFrame(cv_text).T.loc[cv_mean.index]
cv_table

In [ ]:
kandidat = cv_mean.drop(index="Baseline (Dummy)")
best_name = kandidat["roc_auc"].idxmax()
print(f"Model terbaik di CV: {best_name}  (ROC-AUC = {kandidat.loc[best_name, 'roc_auc']:.3f})")

ax = kandidat[["roc_auc", "pr_auc"]].plot.barh(figsize=(8, 3.5), color=["#4C78A8", "#F58518"])
ax.axvline(0.5, ls="--", color="grey", lw=1)
ax.set_xlabel("skor CV (rata-rata)")
ax.set_title("Perbandingan model (garis putus = tebakan acak untuk ROC-AUC)")
plt.tight_layout()
plt.show()

**Cara membaca:** jika beda antar model kecil dibanding simpangan baku (±), anggap performanya setara dan pilih yang lebih sederhana/mudah dijelaskan. Bila semua model berhenti di kisaran skor yang sama, batasnya biasanya ada pada **informasi di dalam data**, bukan pada algoritmanya. Model terbaik otomatis dipilih pada sel berikutnya untuk di-tuning.

## 8. Hyperparameter Tuning

`RandomizedSearchCV` dijalankan **hanya pada model terbaik** dari tahap 7, dengan 5-fold stratified CV di data train dan skor ROC-AUC. Skor CV hasil tuning cenderung sedikit optimistis (karena dipilih dari banyak kombinasi), maka angka yang jujur adalah hasil di test set pada tahap 10.

In [ ]:
param_spaces = {
    "Logistic Regression": {
        "clf__C": loguniform(1e-3, 1e2),
    },
    "Random Forest": {
        "clf__n_estimators": randint(200, 600),
        "clf__max_depth": [None, 4, 6, 8, 12],
        "clf__min_samples_leaf": randint(1, 15),
        "clf__max_features": ["sqrt", "log2", 0.5],
    },
    "Gradient Boosting (Hist)": {
        "clf__learning_rate": loguniform(0.01, 0.3),
        "clf__max_iter": randint(100, 400),
        "clf__max_leaf_nodes": randint(4, 32),
        "clf__min_samples_leaf": randint(10, 60),
        "clf__l2_regularization": loguniform(1e-3, 10),
        "clf__max_depth": [None, 3, 4, 6],
    },
}

base_pipe = Pipeline([("prep", make_preprocessor()), ("clf", clone(models[best_name]))])

search = RandomizedSearchCV(
    base_pipe,
    param_distributions=param_spaces[best_name],
    n_iter=30,
    scoring="roc_auc",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    n_jobs=-1,
    random_state=SEED,
    refit=True,
)
search.fit(X_train, y_train)

print(f"Model yang di-tuning : {best_name}")
print(f"ROC-AUC CV (tuned)   : {search.best_score_:.3f}")
print(f"ROC-AUC CV (default) : {kandidat.loc[best_name, 'roc_auc']:.3f}")
print("Parameter terbaik    :")
for k, v in search.best_params_.items():
    print(f"   {k.replace('clf__', '')}: {round(v, 5) if isinstance(v, float) else v}")

final_pipe = search.best_estimator_

## 9. Pemilihan Threshold (dari Prediksi Out-of-Fold)

Threshold 0,5 belum tentu optimal. Threshold dipilih dari **probabilitas out-of-fold** pada data train (setiap baris diprediksi oleh model yang tidak pernah melihatnya), **bukan dari test set**, supaya evaluasi akhir tetap jujur.

Secara default dipilih threshold yang memaksimalkan **F1**. Di praktik HR, Anda bisa mengubah kriteria ini. Misalnya bila intervensi retensi murah dan kehilangan karyawan mahal, pilih threshold yang menjamin **recall ≥ 0,8**.

In [ ]:
oof_proba = cross_val_predict(
    clone(final_pipe), X_train, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    method="predict_proba", n_jobs=-1,
)[:, 1]

thresholds = np.arange(0.05, 0.96, 0.01)
tabel_thr = pd.DataFrame({
    "threshold": thresholds,
    "precision": [precision_score(y_train, oof_proba >= t, zero_division=0) for t in thresholds],
    "recall": [recall_score(y_train, oof_proba >= t) for t in thresholds],
    "f1": [f1_score(y_train, oof_proba >= t) for t in thresholds],
})

BEST_THRESHOLD = round(float(tabel_thr.loc[tabel_thr["f1"].idxmax(), "threshold"]), 2)
print(f"Threshold terpilih (maks F1 pada OOF): {BEST_THRESHOLD:.2f}")

alt = tabel_thr[tabel_thr["recall"] >= 0.80]
if len(alt):
    print(f"Alternatif recall >= 0.80          : threshold {alt['threshold'].max():.2f} "
          f"(precision {alt.loc[alt['threshold'].idxmax(), 'precision']:.2f})")

fig, ax = plt.subplots(figsize=(8, 4))
for m, c in zip(["precision", "recall", "f1"], ["#4C78A8", "#E45756", "#54A24B"]):
    ax.plot(tabel_thr["threshold"], tabel_thr[m], label=m, color=c)
ax.axvline(BEST_THRESHOLD, ls="--", color="black", lw=1, label=f"terpilih = {BEST_THRESHOLD:.2f}")
ax.set_xlabel("threshold")
ax.set_ylabel("skor (OOF, data train)")
ax.set_title("Trade-off precision–recall terhadap threshold")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Evaluasi Final pada Test Set

Untuk pertama kali data test dipakai, **satu kali saja**. Kita bandingkan dengan baseline dan cek gejala overfitting (train vs CV vs test).

In [ ]:
proba_test = final_pipe.predict_proba(X_test)[:, 1]
proba_train = final_pipe.predict_proba(X_train)[:, 1]
y_te = y_test.to_numpy()

def ringkas_metrik(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred),
        "f1": f1_score(y_true, pred),
    }

metrik_test = {
    "roc_auc": roc_auc_score(y_te, proba_test),
    "pr_auc": average_precision_score(y_te, proba_test),
    **ringkas_metrik(y_te, proba_test, BEST_THRESHOLD),
}

hasil = pd.DataFrame({
    "threshold 0.50": ringkas_metrik(y_te, proba_test, 0.5),
    f"threshold {BEST_THRESHOLD:.2f} (terpilih)": ringkas_metrik(y_te, proba_test, BEST_THRESHOLD),
}).T.round(3)
print("=== Metrik di TEST SET ===")
print(f"ROC-AUC : {metrik_test['roc_auc']:.3f}")
print(f"PR-AUC  : {metrik_test['pr_auc']:.3f}   (baseline PR-AUC = rasio positif = {y_te.mean():.3f})")
print()
print(hasil.to_string())

rng = np.random.default_rng(SEED)
boot = []
for _ in range(1000):
    idx = rng.integers(0, len(y_te), len(y_te))
    if len(np.unique(y_te[idx])) < 2:
        continue
    boot.append(roc_auc_score(y_te[idx], proba_test[idx]))
ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
print(f"\nROC-AUC test 95% CI (bootstrap): [{ci_low:.3f}, {ci_high:.3f}]")

print("\n=== Cek overfitting (ROC-AUC) ===")
print(f"train : {roc_auc_score(y_train, proba_train):.3f}")
print(f"CV    : {search.best_score_:.3f}")
print(f"test  : {metrik_test['roc_auc']:.3f}")

In [ ]:
print(classification_report(
    y_te, (proba_test >= BEST_THRESHOLD).astype(int),
    target_names=["bertahan (0)", "keluar (1)"], digits=3,
))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

RocCurveDisplay.from_predictions(y_te, proba_test, ax=axes[0], name=best_name)
axes[0].plot([0, 1], [0, 1], "--", color="grey", label="acak")
axes[0].set_title("Kurva ROC (test)")
axes[0].legend(loc="lower right")

PrecisionRecallDisplay.from_predictions(y_te, proba_test, ax=axes[1], name=best_name)
axes[1].axhline(y_te.mean(), ls="--", color="grey", label="baseline (acak)")
axes[1].set_title("Kurva Precision–Recall (test)")
axes[1].legend(loc="lower left")

ConfusionMatrixDisplay.from_predictions(
    y_te, (proba_test >= BEST_THRESHOLD).astype(int),
    display_labels=["bertahan", "keluar"], cmap="Blues", ax=axes[2], colorbar=False)
axes[2].set_title(f"Confusion matrix (threshold {BEST_THRESHOLD:.2f})")
axes[2].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
frac_pos, mean_pred = calibration_curve(y_te, proba_test, n_bins=8, strategy="quantile")

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], "--", color="grey", label="terkalibrasi sempurna")
ax.plot(mean_pred, frac_pos, "o-", color="#E45756", label=best_name)
ax.set_xlabel("probabilitas prediksi (rata-rata per bin)")
ax.set_ylabel("proporsi aktual keluar")
ax.set_title("Kurva kalibrasi (test)")
ax.legend()
plt.tight_layout()
plt.show()

**Cara membaca:**
- Jarak antara skor **train**, **CV**, dan **test** yang besar menandakan overfitting. Bila train jauh lebih tinggi daripada test, perkecil kompleksitas model (mis. `max_depth`, `min_samples_leaf`).
- Metrik pada threshold terpilih bisa sedikit lebih rendah/tinggi daripada threshold 0,50 di test set. Itu wajar pada 200 baris; yang penting threshold tidak dipilih dengan melihat test.
- Interval kepercayaan bootstrap menunjukkan seberapa stabil ROC-AUC dengan ukuran test yang hanya 200 baris. Interval lebar berarti angka test tidak boleh dibaca terlalu presisi.
- Kurva kalibrasi yang dekat diagonal berarti probabilitas dapat dibaca literal. Jika jauh dari diagonal, pakai sebagai **skor peringkat risiko** atau kalibrasi dengan `CalibratedClassifierCV`. Dengan 200 baris, kurva ini berisik, jadi baca polanya saja.

## 11. Interpretasi Model

- **Permutation importance** di test set: seberapa turun ROC-AUC bila satu fitur diacak. Ini bekerja untuk model apa pun dan langsung pada fitur asli (bukan kolom hasil one-hot).
- **Partial dependence:** bentuk hubungan fitur numerik utama dengan probabilitas keluar. Kurva yang hampir datar berarti fitur itu nyaris tidak memengaruhi prediksi.

In [ ]:
perm = permutation_importance(
    final_pipe, X_test, y_test, scoring="roc_auc",
    n_repeats=30, random_state=SEED, n_jobs=-1,
)
imp = (pd.DataFrame({"fitur": X_test.columns,
                     "penurunan_auc": perm.importances_mean,
                     "std": perm.importances_std})
       .sort_values("penurunan_auc", ascending=True))

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(imp["fitur"], imp["penurunan_auc"], xerr=imp["std"], color="#4C78A8")
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("penurunan ROC-AUC saat fitur diacak")
ax.set_title("Permutation importance (test set)")
plt.tight_layout()
plt.show()

imp.sort_values("penurunan_auc", ascending=False).head(6).round(4).reset_index(drop=True)

In [ ]:
kontinu = [f for f in NUM_COLS if f != "ada_promosi"]   # fitur biner kurang informatif untuk kurva PDP
top_num = [f for f in imp.sort_values("penurunan_auc", ascending=False)["fitur"] if f in kontinu][:3]

fig, ax = plt.subplots(1, len(top_num), figsize=(5 * len(top_num), 4))
PartialDependenceDisplay.from_estimator(final_pipe, X_train, top_num, ax=ax, grid_resolution=30)
fig.suptitle("Partial dependence — pengaruh fitur numerik utama terhadap probabilitas keluar", y=1.03)
plt.tight_layout()
plt.show()

> **Catatan interpretasi:** importance dan partial dependence menunjukkan **apa yang dipakai model**, bukan **penyebab** turnover. Hubungan seperti lembur → turnover perlu divalidasi lewat konteks bisnis (mis. exit interview) sebelum dijadikan dasar kebijakan.

## 12. Simpan Model, Metadata, dan Contoh Inferensi

Setelah performa divalidasi di test set, model final **dilatih ulang pada seluruh data** (train + test) dengan hyperparameter dan threshold yang sudah dipilih. Data lebih banyak biasanya membuat model sedikit lebih baik. Metrik di metadata tetap berasal dari evaluasi hold-out tahap 10 (model yang dilatih hanya di data train).

Dua berkas disimpan di `models/`:
- `turnover_model.joblib` — pipeline lengkap (preprocessing + model)
- `model_metadata.json` — fitur, threshold, hyperparameter, metrik, versi library

In [ ]:
if REFIT_ON_FULL_DATA:
    model_final = clone(final_pipe).fit(X, y)
    print(f"Model final dilatih ulang pada seluruh data ({len(X)} baris).")
else:
    model_final = final_pipe
    print("Model final = model yang dilatih di data train saja.")

MODEL_PATH = ARTIFACT_DIR / "turnover_model.joblib"
META_PATH = ARTIFACT_DIR / "model_metadata.json"

metadata = {
    "dibuat_pada": datetime.now().isoformat(timespec="seconds"),
    "tujuan": "Prediksi turnover karyawan (1 = keluar)",
    "sumber_data": DATA_PATH.name,
    "jumlah_baris": int(len(df_raw)),
    "rasio_positif": round(float(y.mean()), 4),
    "model": best_name,
    "hyperparameter": {k.replace("clf__", ""): v for k, v in search.best_params_.items()},
    "threshold": BEST_THRESHOLD,
    "refit_seluruh_data": REFIT_ON_FULL_DATA,
    "fitur_numerik": NUM_COLS,
    "fitur_kategorik": CAT_COLS,
    "kolom_mentah_dibutuhkan": [
        "usia", "masa_kerja_bulan", "jam_lembur_bulanan", "gaji_bulanan",
        "tanggal_masuk", "tanggal_promosi", "departemen", "status_kerja", "pendidikan",
    ],
    "tanggal_acuan": str(REF_DATE.date()),
    "metrik_holdout_test": {
        **{k: round(float(v), 4) for k, v in metrik_test.items()},
        "roc_auc_95ci": [round(float(ci_low), 4), round(float(ci_high), 4)],
    },
    "roc_auc_cv_tuned": round(float(search.best_score_), 4),
    "versi": {"python_sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
    "catatan": "Gunakan fungsi buat_fitur() yang sama untuk mengubah data mentah menjadi fitur sebelum predict.",
}

joblib.dump(model_final, MODEL_PATH)
META_PATH.write_text(json.dumps(metadata, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

print("Tersimpan:")
print(f"  {MODEL_PATH}  ({MODEL_PATH.stat().st_size / 1024:.0f} KB)")
print(f"  {META_PATH}")

In [ ]:
model_loaded = joblib.load(MODEL_PATH)
meta_loaded = json.loads(META_PATH.read_text(encoding="utf-8"))


def prediksi_turnover(df_baru: pd.DataFrame, model=model_loaded, meta=meta_loaded) -> pd.DataFrame:
    """Terima data mentah (kolom seperti CSV asli), kembalikan probabilitas & label risiko."""
    perlu = meta["kolom_mentah_dibutuhkan"]
    kurang = [c for c in perlu if c not in df_baru.columns]
    if kurang:
        raise ValueError(f"Kolom wajib belum ada: {kurang}")

    proba = model.predict_proba(buat_fitur(df_baru, pd.Timestamp(meta["tanggal_acuan"])))[:, 1]
    hasil = pd.DataFrame({
        ID_COL: df_baru[ID_COL].values if ID_COL in df_baru.columns else np.arange(len(df_baru)),
        "probabilitas_keluar": proba.round(3),
        "prediksi_keluar": (proba >= meta["threshold"]).astype(int),
    })

    hasil["level_risiko"] = pd.cut(proba, [-0.01, 0.25, 0.50, 1.01], labels=["Rendah", "Sedang", "Tinggi"])
    return hasil


contoh = df_raw.sample(8, random_state=1)
out = prediksi_turnover(contoh)
out["aktual"] = contoh[TARGET].values
print("Model termuat :", meta_loaded["model"], "| threshold =", meta_loaded["threshold"])
out

In [ ]:
p_mem = model_final.predict_proba(X_test)[:, 1]
p_disk = model_loaded.predict_proba(X_test)[:, 1]
print("Prediksi model tersimpan identik dengan model di memori:", np.allclose(p_mem, p_disk))

## Ringkasan

In [ ]:
print("=" * 62)
print("RINGKASAN PIPELINE")
print("=" * 62)
print(f"Data             : {len(df_raw)} karyawan, rasio turnover {y.mean():.1%}")
print(f"Fitur            : {len(FEATURES)} ({len(NUM_COLS)} numerik + {len(CAT_COLS)} kategorik)")
print(f"Model terpilih   : {best_name}")
print(f"ROC-AUC CV       : {search.best_score_:.3f}")
print(f"ROC-AUC test     : {metrik_test['roc_auc']:.3f}  (95% CI {ci_low:.3f}–{ci_high:.3f})")
print(f"PR-AUC test      : {metrik_test['pr_auc']:.3f}  (acak = {y_te.mean():.3f})")
print(f"Threshold        : {BEST_THRESHOLD:.2f} → precision {metrik_test['precision']:.2f}, recall {metrik_test['recall']:.2f}, F1 {metrik_test['f1']:.2f}")
print(f"Fitur terpenting : {', '.join(imp.sort_values('penurunan_auc', ascending=False)['fitur'].head(3))}")
print(f"Model tersimpan  : {MODEL_PATH}")

## Rekomendasi Lanjutan

**Kualitas data & fitur**
- Tambahkan sinyal yang biasanya kuat untuk turnover tapi belum ada di data ini: rating kinerja, skor kepuasan/engagement, jarak tempuh, jumlah cuti, riwayat mutasi, manajer langsung.
- Buat fitur relatif: gaji dibanding median departemen/jabatan/pendidikan (hitung di dalam `Pipeline` agar tidak bocor).
- Bila ada tanggal keluar, ubah formulasi menjadi **survival analysis** (mis. Cox / gradient boosting survival) sehingga waktu ke keluar ikut dimodelkan, bukan hanya ya/tidak.

**Validasi**
- Data ini hanya 1.000 baris, sehingga hasil test sensitif terhadap pembagian data. Perhatikan interval kepercayaan, dan ulangi dengan nested CV bila ingin estimasi lebih stabil.
- Bila data punya dimensi waktu, gunakan **validasi berbasis waktu** (train masa lalu, test masa depan), bukan split acak.
- Pastikan fitur seperti `masa_kerja_bulan` dan `jam_lembur_bulanan` diukur **sebelum** karyawan keluar. Jika diukur sampai tanggal keluar, ada risiko *leakage* temporal.

**Penggunaan & etika**
- Gunakan hasil sebagai **skor prioritas untuk percakapan retensi**, bukan untuk keputusan merugikan individu (PHK, tidak dipromosi).
- `usia` dan `pendidikan` adalah atribut sensitif. Audit performa dan rasio prediksi per kelompok (usia, departemen, status kerja) untuk memastikan tidak ada bias sistematis, dan pertimbangkan mengeluarkan `usia`.

**Operasional**
- Pantau *data drift* (distribusi lembur, gaji) dan performa berkala. Latih ulang tiap 3–6 bulan atau saat performa turun.
- Simpan versi model, data, dan metadata (sudah disiapkan di `model_metadata.json`) agar hasil bisa direproduksi.